# T6 → T8 — xếp hạng lại ColBERT (2 GPU), đánh giá (nếu có nhãn), tạo bài nộp

**Cài đặt notebook:** Accelerator = **GPU T4 ×2**, Internet = **On**.
**Add Input:** `mir-code`, `mir-data`, `mir-runs`.
**Biến ở ô (1):** `RUN` (run cần xếp hạng lại), `K_DOC`/`K_CHUNK` (số tài liệu/đoạn mỗi câu trong bài nộp),
`QRELS` (đường dẫn file nhãn, nếu có — xem `docs/t7_danh_gia.md`).
**Thời gian:** xếp hạng lại 1.200 câu × top-100 mất 27,7 phút trên T4 ×2 (đo 05/10/2026).
**Đầu ra:** `submissions/<tên>.zip`. Tải về từ panel *Output* bên phải (phiên tương tác) hoặc tab *Output* của
version, rồi nộp ở leaderboard.aiguru.com.vn → *My Submissions*. Mỗi lần nộp ghi lại cấu hình và điểm.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây; phần còn lại giống nhau ở mọi notebook.
NB_VERSION = 't6-v2 (05/10/2026)'
RUN = 'structure_G'         # run cần xếp hạng lại (G = Dense+LSR+BM25)
K_DOC, K_CHUNK = 10, 10     # số tài liệu / đoạn mỗi câu hỏi trong bài nộp (giá trị tạm, chưa kiểm chứng)
QRELS = None                # file nhãn đánh giá, vd '/kaggle/input/mir-qrels/qrels.parquet'
OVERRIDES = {'submit.k_doc': K_DOC, 'submit.k_chunk': K_CHUNK}

import glob, os, shutil, sys
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')          # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mir/__init__.py', recursive=True), key=len)
assert hits, 'Chưa thêm dataset mir-code (Add Input → Your Datasets)'
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)                     # luôn chép lại bản mir-code mới nhất
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mir.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, OVERRIDES, corpus='full')
RUNS_IN = locate(f'runs/{RUN}.parquet')
os.makedirs(f'{K.work}/runs', exist_ok=True)
for p in glob.glob(f'{RUNS_IN}/runs/*.parquet') + glob.glob(f'{RUNS_IN}/runs/*.json'):   # chép vào thư mục ghi được
    if not os.path.exists(f'{K.work}/runs/{os.path.basename(p)}'):
        shutil.copy(p, f'{K.work}/runs/')
print('runs:', sorted(os.listdir(f'{K.work}/runs')))

In [ ]:
# (2) T6 — xếp hạng lại top-100 trên 2 GPU (mỗi GPU một nửa số câu hỏi), rồi gộp 2 phần → runs/{RUN}_rr.parquet
cmd = f'python -m mir.rerank --config {K.cfg} --run {K.work}/runs/{RUN}.parquet'
par([f'{cmd} --shard {g}/2 --device cuda:{g}' for g in range(2)], 'T6_rerank')
sh(f'{cmd} --shard 0/2 --device cuda:0')

In [ ]:
# (3) T7 — đánh giá (chỉ khi đã có nhãn): Recall@K, nDCG@10, MRR@10, P/R/F2 → results/
if QRELS:
    sh(f'python -m mir.evaluate --config {K.cfg} --qrels {QRELS} --tables', 'T7_eval')

In [ ]:
# (4) T8 — bài nộp (kiểm tra đủ 1.200 câu, doc_id có trong links_corpus, chunk_text nguyên văn) → submissions/*.zip
NAME = f'{RUN}_rr_d{K_DOC}_c{K_CHUNK}'
sh(f'python -m mir.submit --config {K.cfg} --run {K.work}/runs/{RUN}_rr.parquet --name {NAME}', 'T8_submit')
report('t6')